In [ ]:
# 1) IMPORTS + FILE SETTINGS
import pandas as pd
import numpy as np

file_path = "Historical_Property_Assessments_(Parcel)_20260317.csv"
chunk_size = 100000

dtype_map = {
    'LAND_USE_DESIGNATION': 'string',
    'SUB_PROPERTY_USE': 'string',
    'ADDRESS': 'string',
    'COMM_CODE': 'string',
    'COMM_NAME': 'string',
    'PROPERTY_TYPE': 'string',
    'MULTIPOLYGON': 'string',
    'CPID': 'string'
}

In [ ]:
# 2) FIND THE LAST 7 ROLL YEARS
years = set()

for chunk in pd.read_csv(
    file_path,
    usecols=['ROLL_YEAR'],
    chunksize=chunk_size,
    low_memory=False
):
    years.update(chunk['ROLL_YEAR'].dropna().unique())

last_years = sorted(years)[-7:]
print("Last 7 years:", last_years)

Last 7 years: [2019, 2020, 2021, 2022, 2023, 2024, 2025]


In [ ]:
# 3) BUILD A REDUCED DATASET IN CHUNKS
numeric_cols = [
    'ASSESSED_VALUE', 'RE_ASSESSED_VALUE', 'NR_ASSESSED_VALUE', 'FL_ASSESSED_VALUE',
    'LAND_SIZE_SM', 'LAND_SIZE_SF', 'LAND_SIZE_AC'
]

chunks = []

for chunk in pd.read_csv(
    file_path,
    chunksize=chunk_size,
    low_memory=False,
    dtype=dtype_map
):
    for col in numeric_cols:
        chunk[col] = pd.to_numeric(chunk[col].replace({',': ''}, regex=True), errors='coerce')

    chunk['ASSESSMENT_CLASS'] = (
        chunk['ASSESSMENT_CLASS']
        .astype(str)
        .str.strip()
        .str.upper()
    )

    chunk = chunk[chunk['ROLL_YEAR'].isin(last_years)].copy()

    valid_rolls = chunk.groupby('ROLL_NUMBER')['ASSESSED_VALUE'].transform(lambda x: x.notna().any())
    chunk = chunk[valid_rolls].copy()

    chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)

print("Reduced dataset shape:", df.shape)
print("Years included:", sorted(df['ROLL_YEAR'].dropna().unique()))

Reduced dataset shape: (3952775, 20)
Years included: [2019, 2020, 2021, 2022, 2023, 2024, 2025]


In [ ]:
# 4) SAMPLE 20% OF UNIQUE ROLL NUMBERS
# Sample at the property level, not the row level, so each sampled property
# keeps all of its available years together.

sample_rolls = (
    df['ROLL_NUMBER']
    .dropna()
    .drop_duplicates()
    .sample(frac=0.2, random_state=42)
)

df = df[df['ROLL_NUMBER'].isin(sample_rolls)].copy().reset_index(drop=True)

print(f"Working sample contains {len(df)} rows from {df['ROLL_NUMBER'].nunique()} roll numbers")

Working sample contains 791248 rows from 118693 roll numbers


In [ ]:
# 5) FILL MISSING VALUES WITHIN EACH PROPERTY
# - For columns that should be stable across years, fill missing values using
#   the first non-null value within that property
# - For columns that may change over time, fill missing values using the most
#   recent non-null value within that property

constant_cols = [
    'ADDRESS', 'COMM_CODE', 'COMM_NAME', 'YEAR_OF_CONSTRUCTION',
    'PROPERTY_TYPE', 'LAND_SIZE_SM', 'LAND_SIZE_SF', 'LAND_SIZE_AC',
    'MULTIPOLYGON', 'CPID'
]

variable_cols = [
    'LAND_USE_DESIGNATION', 'SUB_PROPERTY_USE'
]

def fill_property_history(group):
    group = group.sort_values('ROLL_YEAR').copy()

    for col in constant_cols:
        vals = group[col].dropna()
        if not vals.empty:
            group[col] = group[col].fillna(vals.iloc[0])

    for col in variable_cols:
        vals = group[col].dropna()
        if not vals.empty:
            group[col] = group[col].fillna(vals.iloc[-1])

    return group

df = (
    df.groupby('ROLL_NUMBER', group_keys=False)
    .apply(fill_property_history)
    .reset_index(drop=True)
)

C:\Users\chris\AppData\Local\Temp\ipykernel_127288\112077752.py:34: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(fill_property_history)


In [ ]:
# 6) DROP PROPERTIES WITH NO VARIABLE-FIELD INFORMATION
# Keep only properties where at least one of the variable columns contains data
# after the within-property filling step.

has_variable_info = (
    df.groupby('ROLL_NUMBER')[variable_cols]
    .transform(lambda x: x.notna().any())
)

df = df[has_variable_info.any(axis=1)].copy().reset_index(drop=True)

print("After dropping empty variable-field properties:", df.shape)

After dropping empty variable-field properties: (791238, 20)


In [ ]:
# 7) BUILD THE FINAL ANALYSIS DATASET
# - Keep only relevant assessment classes
# - Require positive land size
# - Create VALUE_PER_SM
# - Remove rows where VALUE_PER_SM is missing, zero, or negative

valid_classes = ['RE', 'NR', 'FL']

df_analysis = df[
    df['ASSESSMENT_CLASS'].isin(valid_classes) &
    (df['LAND_SIZE_SM'] > 0)
].copy()

df_analysis['VALUE_PER_SM'] = df_analysis['ASSESSED_VALUE'] / df_analysis['LAND_SIZE_SM']

total_before = len(df_analysis)
mask = df_analysis['VALUE_PER_SM'] > 0
removed = (~mask).sum()

df_analysis = df_analysis[mask].copy().reset_index(drop=True)

print(f"Removed {removed} rows out of {total_before} ({removed / total_before:.2%})")
print("Final shape:", df_analysis.shape)
print("Years included:", sorted(df_analysis['ROLL_YEAR'].unique()))

Removed 1393 rows out of 788939 (0.18%)
Final shape: (787546, 21)
Years included: [2019, 2020, 2021, 2022, 2023, 2024, 2025]


In [ ]:
# 8) SAVE THE FINAL CLEANED SAMPLE
df_analysis.to_csv("calgary_properties_20pct_roll_sample.csv", index=False)
df_analysis.to_parquet("calgary_properties_20pct_roll_sample.parquet", index=False)

In [ ]:
# 9) OPTIONAL CHECKS
print(df_analysis.head())
print(df_analysis.shape)
print(df_analysis['ROLL_YEAR'].value_counts().sort_index())
print(df_analysis['ROLL_NUMBER'].nunique())

   ROLL_YEAR  ROLL_NUMBER        ADDRESS  ASSESSED_VALUE ASSESSMENT_CLASS  \
0       2019      3001609  4715 88 AV NE        509000.0               RE   
1       2020      3001609  4715 88 AV NE        525500.0               RE   
2       2021      3001609  4715 88 AV NE        586500.0               RE   
3       2019      3002102  8825 52 ST NE        591500.0               RE   
4       2020      3002102  8825 52 ST NE        610000.0               RE   

  ASSESSMENT_CLASS_DESCRIPTION  RE_ASSESSED_VALUE  NR_ASSESSED_VALUE  \
0                  Residential           407200.0           101800.0   
1                  Residential           420400.0           105100.0   
2                  Residential           469200.0           117300.0   
3                  Residential           591500.0                NaN   
4                  Residential           610000.0                NaN   

   FL_ASSESSED_VALUE COMM_CODE  ... YEAR_OF_CONSTRUCTION  \
0                NaN       SAD  ...         